In [1]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


#### Read in data and collect tables

In [2]:
person_ids <- read.csv('data/additional_person_ids.csv', header = TRUE)

In [3]:
school_data = "CB_2489.cb_InstitutionHistory"

In [4]:
school_match = "CB_2489.cb_School"

In [5]:
school_link_data = "CB_2489.cb_InstitutionLink"

In [6]:
enrol_data = "CB_2489.cb_Enrollment"

In [7]:
school_match_table <- tbl(con, school_match) |>
    select(SchoolID, URN
           ) 

In [8]:
enrol_table <- tbl(con, enrol_data) |>
    select(person_id, SchoolID, AcademicYear, 
           OnRoll_1, OnRoll_2, OnRoll_3, 
           EnrolStatus, EntryDate, LeavingDate, PartTime
           ) 

In [9]:
school_table <- tbl(con, school_data) |>
    select(InstitutionHistoryID, InstitutionID, AcademicYear,
           InstitutionTypeDesc, URN, SchoolName,
           PhaseOfEducationDesc
           ) 

In [10]:
school_link_table <- tbl(con, school_link_data) |>
    select(InstitutionLinkID, InstitutionHistoryID, AcademicYear,
           person_id
           ) 

In [11]:
school_match_df <- collect(school_match_table)

In [12]:
head(school_match_df)

SchoolID,URN
<dbl>,<dbl>
3978,108234
1935,108274
21112,108314
11201,108189
14610,139568
3644,108238


In [29]:
school_match <- school_match_df |>
    filter(SchoolID %in% c(cohort_schools)) 

In [30]:
head(school_match)

SchoolID,URN
<dbl>,<dbl>
1935,108274
1601,108280
712,108292
1338,108283
749,108275
461,108295


In [14]:
enrol_df <- collect(enrol_table)

In [15]:
school_df <- collect(school_table)

In [16]:
school_link_df <- collect(school_link_table)

In [17]:
school_link_filtered <- school_link_df |>
    filter(person_id %in% person_ids$person_id) 

In [18]:
school_link_filtered

InstitutionLinkID,InstitutionHistoryID,AcademicYear,person_id
<dbl>,<dbl>,<chr>,<chr>


#### Filter enrolments table to target cohorts  

In [19]:
enrol_filtered <- enrol_df |>
    filter(person_id %in% person_ids$person_id) 

In [20]:
enrol_filtered |> arrange(AcademicYear) |> distinct(AcademicYear) |> pull(AcademicYear)

[1] "2005/2006" "2006/2007" "2007/2008" "2008/2009" "2009/2010" "2010/2011"
 [7] "2011/2012" "2012/2013" "2013/2014" "2014/2015" "2015/2016"

In [21]:
enrol_filtered <- enrol_filtered |>
    left_join(person_ids, by = join_by(person_id))

Warning message in left_join(enrol_filtered, person_ids, by = join_by(person_id)):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 28 of `x` matches multiple rows in `y`.
ℹ Row 8501 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


In [22]:
head(enrol_filtered)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>
A75F65A7F2D3B082CD11C6E84BEB92C794F1E36B71BE97CF675ACF317847DCA8,6589,2006/2007,1,1,1,C,2005-01-17,NA,0,2016/2017
35DFCB5D8F021D95483FF24CE410460A7265CE97C3E6D703CA8DAC6E3E333C47,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016
CD414F29155D31984D61A6AFE5652D361C00277FEEFB3794FEFF130EA28F0CBB,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016
868641B54D3700FE1E77718923B99D3A71905DA252D53EB8494816557E111548,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016
3AF775F1DC69FD59BD020B2884CC8265B0F28DD726A1CB98C8F0091B0E3BE67B,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016
0BFA0D59291474ECFA993CECC8B3553CB312919D1F7107A5A04EA234F4D9AEF5,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016


In [23]:
enrol_filtered <- enrol_filtered |>
    filter(NCCIS_ACADYR == '2015/2016' & AcademicYear %in% c('2010/2011','2011/2012','2012/2013','2013/2014','2014/2015') | 
           NCCIS_ACADYR == '2016/2017' & AcademicYear %in% c('2011/2012','2012/2013','2013/2014','2014/2015','2015/2016') 
           )

In [24]:
enrol_filtered |>
    filter(SchoolID == 1)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>
25F5060A19A32BB21DE8653F14808D83ADF8249605EEB64EEC27F5407032D447,1,2012/2013,1,1,1,C,2012-03-07,NA,0,2015/2016
E8395971480E040CEB423297E1D9DCC6BBC9D25E02567A15187154E990F86F7F,1,2013/2014,NA,1,NA,C,2014-01-21,NA,0,2015/2016
25B85E5B37238A30264AE7BFFBA79FB6BB585C93CE11D44E928189AB2A5C3982,1,2013/2014,1,1,NA,C,2012-11-08,NA,0,2016/2017
25B85E5B37238A30264AE7BFFBA79FB6BB585C93CE11D44E928189AB2A5C3982,1,2013/2014,1,1,NA,C,2012-11-08,NA,0,2016/2017
25B85E5B37238A30264AE7BFFBA79FB6BB585C93CE11D44E928189AB2A5C3982,1,2012/2013,1,1,1,C,2012-11-08,NA,0,2016/2017
25B85E5B37238A30264AE7BFFBA79FB6BB585C93CE11D44E928189AB2A5C3982,1,2012/2013,1,1,1,C,2012-11-08,NA,0,2016/2017
F311B88ED8899DAB06BCA8BE02AB751528FC7E2B472E86C8EF72C4B11B8BC76F,1,2013/2014,0,NA,NA,C,2012-10-10,2013-10-04,0,2015/2016
FB839E188C42610761A3BCEFD95B87EF4FA29DAB5311684F1D6BBC7EDE139D7A,1,2012/2013,NA,1,0,C,2013-01-31,NA,0,2015/2016
0D2FC3E992EA35237C6F0D9D0AC60DAD63C93BE61C89F9BE236D217987DFF337,1,2010/2011,NA,1,0,C,2011-02-15,2011-07-22,0,2015/2016


In [25]:
enrol_filtered |> arrange(person_id, AcademicYear)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2011/2012,1,1,1,C,2011-09-05,NA,0,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2012/2013,1,1,1,C,2011-09-05,NA,0,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2013/2014,1,1,1,C,2011-09-05,NA,0,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2014/2015,1,1,1,C,2011-09-05,NA,0,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2015/2016,1,1,1,C,2011-09-05,NA,0,2016/2017
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,12433,2010/2011,1,1,1,C,2010-09-01,NA,0,2015/2016
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,12433,2011/2012,1,1,1,C,2010-09-01,NA,0,2015/2016
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,12433,2012/2013,1,1,1,C,2010-09-01,NA,0,2015/2016
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,12433,2013/2014,1,1,1,C,2010-09-01,NA,0,2015/2016


#### Get list of school IDs in enrollments table

In [26]:
cohort_schools <- enrol_filtered |> arrange(SchoolID) |> distinct(SchoolID) |> pull(SchoolID)

In [27]:
length(cohort_schools)

[1] 2262

In [31]:
head(school_match)

SchoolID,URN
<dbl>,<dbl>
1935,108274
1601,108280
712,108292
1338,108283
749,108275
461,108295


#### Join schools info into school match IDs

In [32]:
head(school_df)

InstitutionHistoryID,InstitutionID,AcademicYear,InstitutionTypeDesc,URN,SchoolName,PhaseOfEducationDesc
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
968,3198,2005/2006,Community school (CY),100222,Craven Park School,Primary
25892,8275,2005/2006,Community school (CY),105771,Hill Top Community Primary School,Primary
40609,11287,2005/2006,Community school (CY),109069,Radstock Infant School,Primary
56451,14508,2005/2006,Independent schools (other) (IND),112448,Holme Park School,Not applicable
123604,28104,2005/2006,Community school (CY),130095,Wembley Primary School,Primary
124903,28374,2005/2006,Further Educational sector college,130541,"Park Lane College, Leeds",16 Plus


In [33]:
school_df$URN <- as.numeric(school_df$URN)

In [48]:
schools_filtered <- school_match |>
    left_join(school_df, by = join_by(URN)) |>
    select(SchoolID, URN, AcademicYear, InstitutionTypeDesc, SchoolName, PhaseOfEducationDesc)

In [49]:
schools_filtered <- schools_filtered |>
    distinct()

In [50]:
head(schools_filtered)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
1935,108274,2005/2006,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2006/2007,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2007/2008,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2008/2009,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2009/2010,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2010/2011,Community School (CY),Hemsworth Arts and Community College,Secondary


In [51]:
schools_filtered |>
    filter(SchoolID == 1935)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
1935,108274,2005/2006,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2006/2007,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2007/2008,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2008/2009,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2009/2010,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2010/2011,Community School (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2011/2012,Community School (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2012/2013,Community School (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2013/2014,Community School (CY),Hemsworth Arts and Community College,Secondary


#### Clean up schools data

In [52]:
# fill NAs in the Phase of Edu col
schools_filtered <- schools_filtered |>
  group_by(URN) |>
  mutate(
    PhaseOfEducationDesc = if_else(
      is.na(PhaseOfEducationDesc),
      first(na.omit(PhaseOfEducationDesc)),
      PhaseOfEducationDesc
    )
  ) |>
  ungroup()

In [53]:
# keep only first row per academic year where multiple per school (e.g. change in school type mid year to academy status)
schools_filtered <- schools_filtered |>
  arrange(AcademicYear) |>
  group_by(SchoolID, URN, AcademicYear) |>
  slice(1) |>
  ungroup()

In [54]:
schools_filtered |> 
    filter(SchoolID == 21)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>


In [55]:
# Check if SchoolID + AcademicYear is unique 
schools_filtered |>
  count(SchoolID, AcademicYear) |>
  filter(n > 1)  # problem rows

SchoolID,AcademicYear,n
<dbl>,<chr>,<int>


#### clean up institution types

In [60]:
schools_filtered |>
    distinct(InstitutionTypeDesc)

InstitutionTypeDesc
<chr>
Community school
Voluntary aided school
Foundation school
Academy
Academy sponsor led
Voluntary controlled school
Community special school
Community hospital school
Non-maintained special school


In [57]:
schools_filtered <- schools_filtered |>
  mutate(InstitutionTypeDesc = str_replace_all(InstitutionTypeDesc, "û", "-"))

In [58]:
schools_filtered <- schools_filtered |>
  mutate(InstitutionTypeDesc = InstitutionTypeDesc |> 
           str_trim())

In [59]:
schools_filtered <- schools_filtered |>
    mutate(InstitutionTypeDesc = case_when(
            InstitutionTypeDesc %in% c("Community school (CY)", "Community School (CY)") ~ 'Community school',
        
        InstitutionTypeDesc %in% c("Voluntary aided school (VA)", "Voluntary Aided School (VA)") ~ 'Voluntary aided school',
        
        InstitutionTypeDesc %in% c("Foundation School (FD)", "Foundation school (FD)") ~ 'Foundation school',
        
        InstitutionTypeDesc %in% c("Voluntary controlled school (VC)", "Voluntary Controlled School (VC)") ~ 'Voluntary controlled school',
        
        InstitutionTypeDesc %in% c("Academy (AC)", "Academy Schhol", "Academy Free Schools / Consortia") ~ 'Academy',
        
        InstitutionTypeDesc %in% c("Academy Sponsor Led (AC)", "Academy - Sponsor Led Mainstream (AC)") ~ 'Academy sponsor led',
        
        InstitutionTypeDesc %in% c("City technology college (CTC)", "City technology College (CTC)", "City Technology College (CTC)") ~ 'City technology college',
        
        InstitutionTypeDesc %in% c("Community special school (CYS)", "Community Special School (CYS)") ~ 'Community special school',
        
        InstitutionTypeDesc %in% c("Special school not maintained by LEA (NMSS)", "Non-Maintained Special School (NMSS)") ~ 'Non-maintained special school',
        
        InstitutionTypeDesc %in% c("Foundation special school (FDS)", "Foundation Special School (FDS)") ~ 'Foundation special school',
        
        InstitutionTypeDesc %in% c("Community hospital school (CYH)") ~ 'Community hospital school',
        
        InstitutionTypeDesc %in% c("Academy Converters Mainstream", "Academy Converter - Mainstream (ACC)") ~ 'Academy converter',
        
        InstitutionTypeDesc %in% c("Free School - Mainstream (F)", "Community School (CY)") ~ 'Free school',
        
        InstitutionTypeDesc %in% c("Academy - Converter Special School (ACCS)", "Academy - Sponsor Led Special School (ACS)") ~ 'Academy special school',
        
        InstitutionTypeDesc %in% c("Pupil referral unit", "Pupil Referral Unit", 
                                   'Pupil Referral Unit (PRU)', 'Academy - Converter AP (ACCAP)',
                                  'Free School - Alternative Provision', 'Academy - Sponsor Led AP (ACAP)',
                                  'Free School - Alternative Provision (FAP)') ~ 'AP/PRU',
        
        InstitutionTypeDesc %in% c("Free School - Studio School (FSS)", 'Free School - Studio School') ~ 'Studio school',
        InstitutionTypeDesc %in% c("Free School - UTC (FUTC)", "Free School - UTC") ~ 'University technical college',
        InstitutionTypeDesc %in% c("Free School - Studio School (FSS)") ~ 'Studio school',
        InstitutionTypeDesc %in% c("Free School - 16-19 (F1619)") ~ 'Free school (16-19)',
        InstitutionTypeDesc %in% c("Free School - Special (FS)") ~ 'Free special school (16-19)',
    TRUE ~ InstitutionTypeDesc)
           )
        
        

#### check for schools changing name

In [61]:
schools_filtered <- schools_filtered |>
  mutate(SchoolName = gsub("[[:punct:]]", "", SchoolName))

In [62]:
schools_filtered <- schools_filtered |>
  mutate(SchoolName = SchoolName |> 
           str_trim())

In [63]:
schools_filtered |>
    group_by(URN) |> 
    filter(n_distinct(SchoolName) > 1) |> 
    arrange(URN, SchoolName)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
1512,100182,2012/2013,Community school,Eltham Hill School,Secondary
1512,100182,2013/2014,Community school,Eltham Hill School,Secondary
1512,100182,2014/2015,Community school,Eltham Hill School,Secondary
1512,100182,2015/2016,Community school,Eltham Hill School,Secondary
1512,100182,2016/2017,Community school,Eltham Hill School,Secondary
1512,100182,2017/2018,Community school,Eltham Hill School,Secondary
1512,100182,2018/2019,Community school,Eltham Hill School,Secondary
1512,100182,2019/2020,Community school,Eltham Hill School,Secondary
1512,100182,2020/2021,Community school,Eltham Hill School,Secondary


In [64]:
latest_names <- schools_filtered |>
    group_by(URN) |>
    slice_max(AcademicYear, n = 1, with_ties = FALSE) |>
    select(URN, SchoolName) |>
    rename(LatestSchoolName = SchoolName)

In [65]:
head(latest_names)

URN,LatestSchoolName
<dbl>,<chr>
100049,Haverstock School
100053,Acland Burghley School
100182,Eltham Hill School
100187,Woolwich Polytechnic School for Boys
100189,Crown Woods College
100190,Thomas Tallis School


In [474]:
latest_names |>
    filter(URN == 107302)

URN,LatestSchoolName
<dbl>,<chr>
107302,All Saints CofE Primary School


In [66]:
schools_filtered <- schools_filtered |>
    left_join(latest_names, by = "URN") |>
    mutate(SchoolName = coalesce(LatestSchoolName, SchoolName)) |>
    select(-LatestSchoolName)

In [67]:
head(schools_filtered)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
1,108056,2005/2006,Community school,City of Leeds School,Secondary
1,108056,2006/2007,Community school,City of Leeds School,Secondary
1,108056,2007/2008,Community school,City of Leeds School,Secondary
1,108056,2008/2009,Community school,City of Leeds School,Secondary
1,108056,2009/2010,Community school,City of Leeds School,Secondary
1,108056,2010/2011,Community school,City of Leeds School,Secondary


#### fuzzy match for slightly different name spellings

In [243]:
install.packages('stringdist')

Installing package into ‘/home/jupyter/.R/library’
(as ‘lib’ is unspecified)



In [244]:
library(stringdist)

In [284]:
school_names <- unique(schools_filtered$SchoolName)

In [285]:
dist_matrix <- stringdistmatrix(school_names, school_names, method = "jw")

In [286]:
# Find likely duplicates based on low distance
fuzzy_matches <- which(dist_matrix < 0.1 & dist_matrix > 0, arr.ind = TRUE)

In [301]:
enrol_final |>
    filter(SchoolName == 'Queen Elizabeths School')

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc,LatestURN
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>
BB1C7F4FC50A60FF728A48C7F2A3C4496F2E6765FA62119EBBF5ABB6FB133735,16226,2015/2016,1,1,1,C,2015-09-01,NA,0,2018/2019,141526,Academy converter,Queen Elizabeths School,Secondary,141526
BB1C7F4FC50A60FF728A48C7F2A3C4496F2E6765FA62119EBBF5ABB6FB133735,16226,2016/2017,1,1,1,C,2015-09-01,NA,0,2018/2019,141526,Academy converter,Queen Elizabeths School,Secondary,141526
BB1C7F4FC50A60FF728A48C7F2A3C4496F2E6765FA62119EBBF5ABB6FB133735,16226,2017/2018,1,1,1,C,2015-09-01,NA,0,2018/2019,141526,Academy converter,Queen Elizabeths School,Secondary,141526
D5BC71328A6A3D4C7F394FF461AB2209C88B7569BF0CFFDA0A73A87D0AB8607C,16226,2015/2016,1,0,NA,C,2015-09-02,2016-02-26,0,2018/2019,141526,Academy converter,Queen Elizabeths School,Secondary,141526


In [303]:
enrol_final |>
    filter(SchoolName == "Queen Elizabeths Grammar School")

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc,LatestURN
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>
5E067A183AF6A1DFCA5B9A38548CB42F4131EE05E131A525F083F605CCFADB55,16035,2017/2018,1,1,1,C,2014-09-22,NA,0,2018/2019,141165,Free school,Queen Elizabeths Grammar School,NA,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2013/2014,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2016/2017,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
5E067A183AF6A1DFCA5B9A38548CB42F4131EE05E131A525F083F605CCFADB55,16035,2014/2015,1,1,1,C,2014-09-22,NA,0,2018/2019,141165,Free school,Queen Elizabeths Grammar School,NA,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2014/2015,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2015/2016,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
5E067A183AF6A1DFCA5B9A38548CB42F4131EE05E131A525F083F605CCFADB55,16035,2015/2016,1,1,1,C,2014-09-22,NA,0,2018/2019,141165,Free school,Queen Elizabeths Grammar School,NA,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2012/2013,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
5E067A183AF6A1DFCA5B9A38548CB42F4131EE05E131A525F083F605CCFADB55,16035,2016/2017,1,1,1,C,2014-09-22,NA,0,2018/2019,141165,Free school,Queen Elizabeths Grammar School,NA,136972


In [287]:
tibble(
  name1 = school_names[fuzzy_matches[, 1]],
  name2 = school_names[fuzzy_matches[, 2]],
  distance = dist_matrix[fuzzy_matches]
) %>%
  distinct() %>%
  arrange(distance)

name1,name2,distance
<chr>,<chr>,<dbl>
Queen Elizabeths School,Queen Elizabeth School,0.01449275
Queen Elizabeth School,Queen Elizabeths School,0.01449275
Hinde House 216 School,Hinde House 316 School,0.03030303
Hinde House 316 School,Hinde House 216 School,0.03030303
Bedale High School,Beal High School,0.03703704
Forest Hall School,Forest Hill School,0.03703704
Beal High School,Bedale High School,0.03703704
Forest Hill School,Forest Hall School,0.03703704
Brookfield School,Broomfield School,0.03921569


In [ ]:
#school_names_clean <- schools_filtered |>
#    mutate(SchoolName = case_when(
#            SchoolName %in% c("Hinde House 216 School", "Hinde House 316 School") ~ 'Hinde House 2-16 School',
#        SchoolName %in% c("St Josephs Catholic College", "St Bedes and St Josephs Catholic College") ~ 'St Bedes and St Josephs Catholic College',
#        SchoolName %in% c("Oakbank School", "Beckfoot Oakbank") ~ 'Beckfoot Oakbank',
#        SchoolName %in% c("Tong High School", "Tong Leadership Academy") ~ 'Tong Leadership Academy',
#        SchoolName %in% c("St Anselms Catholic School", "St Anselms Catholic School Canterbury") ~ 'Hinde House 2-16 School',

In [307]:
enrol_final |>
    filter(SchoolName == "St Anselms Catholic School")

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc,LatestURN
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>
329745E3565E94BD632FA7AB0F347C5E1C5B5DDE0E32304B542436108D440D9F,1505,2012/2013,NA,1,1,C,2013-05-07,NA,0,2017/2018,118918,Voluntary aided school,St Anselms Catholic School,Secondary,118918
329745E3565E94BD632FA7AB0F347C5E1C5B5DDE0E32304B542436108D440D9F,1505,2013/2014,1,1,1,M,2013-05-07,NA,0,2017/2018,118918,Voluntary aided school,St Anselms Catholic School,Secondary,118918
296E09A08A59BA0C6C6491466160F8E03009E1F05A3F75C0ADA1D848F541F631,1505,2013/2014,1,1,1,C,2013-09-03,NA,0,2018/2019,118918,Voluntary aided school,St Anselms Catholic School,Secondary,118918


#### Decision - treat academisation as new school - DfE do as asign new URN

#### IGNORE check for schools changing URN because of becoming academies etc... 

In [ ]:
#schools_filtered |>
    group_by(SchoolName) |> 
    filter(n_distinct(URN) > 1) |> 
    arrange(SchoolName, URN)

In [289]:
#latest_urn <- schools_filtered |>
    group_by(SchoolName) |>
    slice_max(AcademicYear, n = 1, with_ties = FALSE) |>
    select(URN, SchoolName) |>
    rename(LatestURN = URN)

In [308]:
#head(latest_urn)

In [291]:
#schools_filtered <- schools_filtered |>
  left_join(latest_urn, by = "SchoolName")

In [329]:
schools_filtered |>
    group_by(SchoolName) |> 
    filter(n_distinct(URN) > 1) |> 
    arrange(SchoolName, URN)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
10414,135622,2007/2008,Academy,Academy 360,Not applicable
10414,135622,2008/2009,Academy,Academy 360,Not applicable
10414,135622,2009/2010,Academy,Academy 360,Not applicable
10414,135622,2010/2011,Academy sponsor led,Academy 360,Not applicable
10414,135622,2011/2012,Academy sponsor led,Academy 360,Not applicable
10414,135622,2012/2013,Academy sponsor led,Academy 360,Not applicable
10414,135622,2013/2014,Academy sponsor led,Academy 360,Secondary
10414,135622,2014/2015,Academy sponsor led,Academy 360,Not applicable
10414,135622,2015/2016,Academy sponsor led,Academy 360,Not applicable


#### Save school info to csv

In [68]:
write.csv(schools_filtered, "data/additional_schools_filtered.csv", row.names = FALSE)

#### Join matched schools data to enrolments table

In [69]:
enrol_final <- enrol_filtered |>
    left_join(schools_filtered, by = join_by(SchoolID, AcademicYear))

In [70]:
enrol_final |>
    arrange(person_id, AcademicYear)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2011/2012,1,1,1,C,2011-09-05,NA,0,2016/2017,136664,Academy converter,Skipton Girls High School,Secondary
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2012/2013,1,1,1,C,2011-09-05,NA,0,2016/2017,136664,Academy converter,Skipton Girls High School,Secondary
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2013/2014,1,1,1,C,2011-09-05,NA,0,2016/2017,136664,Academy converter,Skipton Girls High School,Secondary
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2014/2015,1,1,1,C,2011-09-05,NA,0,2016/2017,136664,Academy converter,Skipton Girls High School,Secondary
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,12506,2015/2016,1,1,1,C,2011-09-05,NA,0,2016/2017,136664,Academy converter,Skipton Girls High School,Secondary
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,12433,2010/2011,1,1,1,C,2010-09-01,NA,0,2015/2016,136736,Academy converter,South Craven School,Secondary
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,12433,2011/2012,1,1,1,C,2010-09-01,NA,0,2015/2016,136736,Academy converter,South Craven School,Secondary
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,12433,2012/2013,1,1,1,C,2010-09-01,NA,0,2015/2016,136736,Academy converter,South Craven School,Secondary
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,12433,2013/2014,1,1,1,C,2010-09-01,NA,0,2015/2016,136736,Academy converter,South Craven School,Secondary


#### check how many school moves in the data

In [71]:
id_lists <- enrol_final |>
    select(person_id, SchoolName)

In [72]:
#id_lists

In [73]:
# n = 85,423
head(id_lists)

person_id,SchoolName
<chr>,<chr>
35DFCB5D8F021D95483FF24CE410460A7265CE97C3E6D703CA8DAC6E3E333C47,Dixons City Academy
CD414F29155D31984D61A6AFE5652D361C00277FEEFB3794FEFF130EA28F0CBB,Dixons City Academy
868641B54D3700FE1E77718923B99D3A71905DA252D53EB8494816557E111548,Dixons City Academy
3AF775F1DC69FD59BD020B2884CC8265B0F28DD726A1CB98C8F0091B0E3BE67B,Dixons City Academy
0BFA0D59291474ECFA993CECC8B3553CB312919D1F7107A5A04EA234F4D9AEF5,Dixons City Academy
8E783AF3FBE68119322CB54C141558117F093036F6CCA94259A8B870A8A07F73,Dixons City Academy


In [74]:
distinct(id_lists)

person_id,SchoolName
<chr>,<chr>
35DFCB5D8F021D95483FF24CE410460A7265CE97C3E6D703CA8DAC6E3E333C47,Dixons City Academy
CD414F29155D31984D61A6AFE5652D361C00277FEEFB3794FEFF130EA28F0CBB,Dixons City Academy
868641B54D3700FE1E77718923B99D3A71905DA252D53EB8494816557E111548,Dixons City Academy
3AF775F1DC69FD59BD020B2884CC8265B0F28DD726A1CB98C8F0091B0E3BE67B,Dixons City Academy
0BFA0D59291474ECFA993CECC8B3553CB312919D1F7107A5A04EA234F4D9AEF5,Dixons City Academy
8E783AF3FBE68119322CB54C141558117F093036F6CCA94259A8B870A8A07F73,Dixons City Academy
41174C3DD06AC89C1538607F536905DD039451D7609845A86B07DFDEC9942FE7,Dixons City Academy
1DE429B6E3EFF9706C8619FED25B2946012BEB3801945340B5346526CBE0D0DB,Dixons City Academy
84F3FA0E59882E7FE2760EE3707F05A31F3455C34F07E1506DA4023D0B1D493B,Dixons City Academy


In [486]:
# n = 24,688
id_lists_distinct <- distinct(id_lists)

In [487]:
person_counts <- table(id_lists_distinct$person_id)
summary <- table(person_counts)

In [488]:
as.data.frame(summary)

person_counts,Freq
<fct>,<int>
1,10792
2,5050
3,808
4,219
5,70
6,18
7,2
8,3


#### Explore high school count ids

In [489]:
enrol_final |> 
    filter(person_id == '38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F') |>
    arrange(AcademicYear, EntryDate)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,1562,2012/2013,1,1,1,C,2012-09-03,NA,0,2017/2018,107440,Foundation school,Hanson School,Secondary
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,1562,2013/2014,1,0,0,C,2012-09-03,2014-04-03,0,2017/2018,107440,Foundation school,Hanson School,Secondary
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,15223,2013/2014,0,NA,NA,NA,2013-09-24,2013-12-13,0,2017/2018,133411,AP/PRU,Bradford Central PRU,Not applicable
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,344,2013/2014,NA,1,1,C,2014-04-04,NA,0,2017/2018,107563,Community school,Sowerby Bridge High School,Secondary
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,344,2014/2015,1,0,0,M,2014-04-04,2015-02-23,0,2017/2018,107563,Community school,Sowerby Bridge High School,Secondary
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,14447,2014/2015,0,NA,NA,S,2014-10-02,2014-11-14,0,2017/2018,133693,AP/PRU,Calderdale PRU,Not applicable
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,14447,2015/2016,1,1,1,C,2014-10-02,NA,0,2017/2018,133693,AP/PRU,Calderdale PRU,Not applicable
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,14447,2016/2017,1,1,0,C,2014-10-02,2017-06-30,0,2017/2018,133693,AP/PRU,Calderdale PRU,Not applicable


#### create a table of person_id and school count

In [490]:
# create a new col with count to match with destinations
school_count <- id_lists_distinct %>%
  group_by(person_id) %>%
  mutate(n_schools = n()) %>%
  ungroup()

In [491]:
head(school_count)

person_id,SchoolName,n_schools
<chr>,<chr>,<int>
F65B08C0ABA0DF84D9BD65F7FED88E1D5DAD42AAB47392CE2D9543AED4E245DD,South Craven School,2
F97D233CF64E3EAC93369EAAA28D9E5193785F4BA369A53A716A1B453B9FDDA2,Appleton Academy,3
3E4432F9D7005D74AB24D68A875EC38A8DA0234A2D67666F935999C059682696,Gloucester Academy,2
823AECF1EFC7F8CD963B2B16D2C886922CD9F7C0CA4B478BD07E86F163685822,Endeavour High School,2
18746FF26D6AC95D757A4B4CB10B01DE6F041115B43ECC8BE0C4A0AEF575224B,Coop Academy Leeds,1
0D5BABE74A35CC1F64DB2C1F97898AD55B2DF12975FE992FE28487B958392B33,Appleton Academy,2


# Pivot Long for terms then Pivot Wide for years

In [75]:
head(enrol_final)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
35DFCB5D8F021D95483FF24CE410460A7265CE97C3E6D703CA8DAC6E3E333C47,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
CD414F29155D31984D61A6AFE5652D361C00277FEEFB3794FEFF130EA28F0CBB,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
868641B54D3700FE1E77718923B99D3A71905DA252D53EB8494816557E111548,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
3AF775F1DC69FD59BD020B2884CC8265B0F28DD726A1CB98C8F0091B0E3BE67B,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
0BFA0D59291474ECFA993CECC8B3553CB312919D1F7107A5A04EA234F4D9AEF5,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
8E783AF3FBE68119322CB54C141558117F093036F6CCA94259A8B870A8A07F73,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable


In [76]:
school_per_term <- enrol_final |>
    select(-c(SchoolID, PartTime, SchoolName, InstitutionTypeDesc, PhaseOfEducationDesc))

In [77]:
head(school_per_term)

person_id,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>
35DFCB5D8F021D95483FF24CE410460A7265CE97C3E6D703CA8DAC6E3E333C47,2010/2011,1,1,1,C,2010-08-27,NA,2015/2016,130909
CD414F29155D31984D61A6AFE5652D361C00277FEEFB3794FEFF130EA28F0CBB,2010/2011,1,1,1,C,2010-08-27,NA,2015/2016,130909
868641B54D3700FE1E77718923B99D3A71905DA252D53EB8494816557E111548,2010/2011,1,1,1,C,2010-08-27,NA,2015/2016,130909
3AF775F1DC69FD59BD020B2884CC8265B0F28DD726A1CB98C8F0091B0E3BE67B,2010/2011,1,1,1,C,2010-08-27,NA,2015/2016,130909
0BFA0D59291474ECFA993CECC8B3553CB312919D1F7107A5A04EA234F4D9AEF5,2010/2011,1,1,1,C,2010-08-27,NA,2015/2016,130909
8E783AF3FBE68119322CB54C141558117F093036F6CCA94259A8B870A8A07F73,2010/2011,1,1,1,C,2010-08-27,NA,2015/2016,130909


C = Current (single registration at this school)
G = Guest (pupil not registered at this school but attending some lessons or sessions)
M = Current Main (dual registration)
S = Current Subsidiary (dual registration)
F = FE College (since 2014/15)
O = Other provider (since 2014/15)

In [78]:
school_per_term |> distinct(EnrolStatus) |> pull(EnrolStatus)

[1] "C" NA  "S" "M" "O" "F"

In [496]:
school_per_term |> 
    filter(person_id == '56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB' ) |>
    arrange(AcademicYear, EntryDate)

person_id,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2012/2013,0,NA,NA,C,2012-09-03,2012-10-05,2017/2018,107366
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2012/2013,1,1,1,C,2012-10-04,NA,2017/2018,135367
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2013/2014,1,1,1,C,2012-10-04,NA,2017/2018,135367
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2014/2015,1,1,1,M,2012-10-04,NA,2017/2018,135367
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2014/2015,0,NA,NA,S,2014-09-26,2014-11-27,2017/2018,107350
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2014/2015,1,0,NA,C,2014-12-10,2015-02-13,2017/2018,133411
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2015/2016,0,NA,NA,M,2012-10-04,2015-11-06,2017/2018,135367
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2015/2016,1,1,1,O,2015-11-09,NA,2017/2018,135732
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2016/2017,1,1,0,C,2015-11-09,2017-06-30,2017/2018,135732


In [497]:
enrol_final |> 
    filter(person_id == '56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB' ) |>
    arrange(AcademicYear, EntryDate)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,1716,2012/2013,0,NA,NA,C,2012-09-03,2012-10-05,0,2017/2018,107366,Foundation school,Tong High School,Secondary
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2012/2013,1,1,1,C,2012-10-04,NA,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2013/2014,1,1,1,C,2012-10-04,NA,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Secondary
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2014/2015,1,1,1,M,2012-10-04,NA,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,1340,2014/2015,0,NA,NA,S,2014-09-26,2014-11-27,0,2017/2018,107350,Foundation school,Buttershaw Business and Enterprise College,Secondary
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,15223,2014/2015,1,0,NA,C,2014-12-10,2015-02-13,0,2017/2018,133411,AP/PRU,Bradford Central PRU,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2015/2016,0,NA,NA,M,2012-10-04,2015-11-06,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,15167,2015/2016,1,1,1,O,2015-11-09,NA,0,2017/2018,135732,AP/PRU,Bradford District PRU,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,15167,2016/2017,1,1,0,C,2015-11-09,2017-06-30,0,2017/2018,135732,AP/PRU,Bradford District PRU,Not applicable


#### pivot long

In [79]:
pivot_long <- school_per_term %>%
  pivot_longer(
    cols = starts_with("OnRoll_"),            
    names_to = "Term",                        # new column name for term
    values_to = "OnRoll",                     # values (0, 1, or NA)
    names_prefix = "OnRoll_"                  # remove this prefix from 'Term'
  )

In [80]:
pivot_long |>
    arrange(person_id, AcademicYear, Term)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,1,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,2,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,3,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,1,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,2,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,3,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2013/2014,C,2011-09-05,NA,2016/2017,136664,1,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2013/2014,C,2011-09-05,NA,2016/2017,136664,2,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2013/2014,C,2011-09-05,NA,2016/2017,136664,3,1


In [81]:
pivot_long |>
  group_by(person_id, AcademicYear, Term) |>
  filter(n() > 1) |>
  arrange(person_id, AcademicYear, Term)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2011/2012,C,2011-09-06,NA,2016/2017,107440,1,1
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2011/2012,C,2011-09-06,NA,2016/2017,107440,1,1
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2011/2012,C,2011-09-06,NA,2016/2017,107440,2,1
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2011/2012,C,2011-09-06,NA,2016/2017,107440,2,1
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2011/2012,C,2011-09-06,NA,2016/2017,107440,3,1
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2011/2012,C,2011-09-06,NA,2016/2017,107440,3,1
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2012/2013,C,2011-09-06,NA,2016/2017,107440,1,1
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2012/2013,C,2011-09-06,NA,2016/2017,107440,1,1
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2012/2013,C,2011-09-06,NA,2016/2017,107440,2,1


In [82]:
pivot_long_clean <- pivot_long |>
    group_by(person_id, AcademicYear, Term) |>
    filter(n() > 1) |>
    mutate(
        # prioritise the record with onroll == T
        onroll_priority = ifelse(OnRoll == "1", 1, 0),
        # prioritise the record with C enrolment, then M enrolment status
        enrol_priority = case_when(
          EnrolStatus == "C" ~ 3,
          EnrolStatus == "M" ~ 2,
          TRUE ~ 1
            ),
        # combine
        priority_score = onroll_priority * 10 + enrol_priority
      ) |>
    # keep single highest priority record
    arrange(desc(priority_score)) |>
    slice_head(n = 1) |>
    ungroup()

In [83]:
non_duplicates <- pivot_long |>
  group_by(person_id, AcademicYear, Term) |>
  filter(n() == 1) |>
  ungroup()

In [88]:
pivot_long_clean <- bind_rows(pivot_long_clean, non_duplicates) |>
  arrange(person_id, AcademicYear, Term) |>
    distinct()

In [85]:
# n = 246807

In [89]:
pivot_long_clean |>
  #group_by(person_id, AcademicYear, Term) |>
  #filter(n() > 1) |>
  arrange(person_id, AcademicYear, Term)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll,onroll_priority,enrol_priority,priority_score
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,1,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,2,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,3,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,1,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,2,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,3,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2013/2014,C,2011-09-05,NA,2016/2017,136664,1,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2013/2014,C,2011-09-05,NA,2016/2017,136664,2,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2013/2014,C,2011-09-05,NA,2016/2017,136664,3,1,NA,NA,NA


In [87]:
head(pivot_long_clean)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll,onroll_priority,enrol_priority,priority_score
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,1,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,2,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011/2012,C,2011-09-05,NA,2016/2017,136664,3,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,1,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,2,1,NA,NA,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012/2013,C,2011-09-05,NA,2016/2017,136664,3,1,NA,NA,NA


In [90]:
df_for_pivot_wide <- pivot_long_clean |>
    select(c(person_id, NCCIS_ACADYR, URN, AcademicYear, Term))

In [91]:
head(df_for_pivot_wide)

person_id,NCCIS_ACADYR,URN,AcademicYear,Term
<chr>,<chr>,<dbl>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,136664,2011/2012,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,136664,2011/2012,2
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,136664,2011/2012,3
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,136664,2012/2013,1
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,136664,2012/2013,2
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,136664,2012/2013,3


In [92]:
# Save as csv 
write.csv(df_for_pivot_wide, "data/additional_school_urn_enrollment_perterm.csv", row.names = FALSE)

In [415]:
#enrol_final
pivot_long_clean |> 
    filter(person_id == '004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894' ) |>
    arrange(AcademicYear)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll,onroll_priority,enrol_priority,priority_score
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2012/2013,C,2012-09-04,NA,2017/2018,107441,1,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2012/2013,C,2012-09-04,NA,2017/2018,107441,2,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2012/2013,C,2012-09-04,NA,2017/2018,107441,3,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,1,1,1,3,13
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,2,1,1,3,13
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,3,1,1,3,13
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2014/2015,S,2014-07-07,NA,2017/2018,107428,1,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2014/2015,S,2014-07-07,NA,2017/2018,107428,2,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2014/2015,S,2014-07-07,NA,2017/2018,107428,3,1,NA,NA,NA


#### Pivot Wide

In [93]:
pivot_wide <- df_for_pivot_wide |>
    arrange(person_id, AcademicYear, Term) |>
    mutate(Year_Term = paste0("Y", AcademicYear, "_", "Term", Term),
          URN = as.character(URN)) |>
    select(person_id, NCCIS_ACADYR, Year_Term, URN) |>
    pivot_wider(
        names_from = Year_Term,
        values_from = URN,
        values_fill = NA
      )

In [94]:
pivot_wide

person_id,NCCIS_ACADYR,Y2011/2012_Term1,Y2011/2012_Term2,Y2011/2012_Term3,Y2012/2013_Term1,Y2012/2013_Term2,Y2012/2013_Term3,Y2013/2014_Term1,Y2013/2014_Term2,Y2013/2014_Term3,Y2014/2015_Term1,Y2014/2015_Term2,Y2014/2015_Term3,Y2015/2016_Term1,Y2015/2016_Term2,Y2015/2016_Term3,Y2010/2011_Term1,Y2010/2011_Term2,Y2010/2011_Term3
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,136664,136664,136664,136664,136664,136664,136664,136664,136664,136664,136664,136664,136664,136664,136664,NA,NA,NA
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2015/2016,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,NA,NA,NA,136736,136736,136736
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,2015/2016,130909,130909,130909,130909,130909,130909,130909,130909,130909,130909,130909,130909,NA,NA,NA,130909,130909,130909
00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,2016/2017,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,136736,NA,NA,NA
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2016/2017,107440,107440,107440,107440,107440,107440,107440,107440,107440,107440,107440,107440,107440,107440,107440,NA,NA,NA
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2016/2017,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,NA,NA,NA
001F42538A9E6E496F13CD05B5E9FE91EB3B20F0CEBECA1B76CC5E7151B7A0D4,2015/2016,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,NA,NA,NA,107395,107395,107395
00211975518FD5336E43850D59B5F5DDA7CD98B212760111CB40528BDECF3B1B,2015/2016,107391,107391,107391,NA,NA,NA,135732,135732,135732,135732,135732,135732,NA,NA,NA,107391,107391,107391
00237E9C21696BB29409630166F2D9CCEC24B83FB120AEDD878DDD54C6516B3D,2015/2016,121716,121716,121716,121716,121716,121716,121716,121716,121716,121716,121716,121716,NA,NA,NA,121716,121716,121716


#### Save as CSV

In [95]:
write.csv(pivot_long, "data/additional_enrolment_by_term.csv", row.names = FALSE)

In [96]:
write.csv(pivot_long_clean, "data/additional_enrolment_by_term_clean.csv", row.names = FALSE)

In [97]:
write.csv(pivot_wide, "data/additional_enrolment_wide.csv", row.names = FALSE)

In [98]:
write.csv(enrol_final, "data/additional_enrol_final.csv", row.names = FALSE)

In [100]:
#write.csv(school_count, "data/additional_school_count.csv", row.names = FALSE)

# extract from full enrolment tables whether ever attended AP/PRU

In [101]:
head(enrol_final)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
35DFCB5D8F021D95483FF24CE410460A7265CE97C3E6D703CA8DAC6E3E333C47,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
CD414F29155D31984D61A6AFE5652D361C00277FEEFB3794FEFF130EA28F0CBB,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
868641B54D3700FE1E77718923B99D3A71905DA252D53EB8494816557E111548,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
3AF775F1DC69FD59BD020B2884CC8265B0F28DD726A1CB98C8F0091B0E3BE67B,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
0BFA0D59291474ECFA993CECC8B3553CB312919D1F7107A5A04EA234F4D9AEF5,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable
8E783AF3FBE68119322CB54C141558117F093036F6CCA94259A8B870A8A07F73,125,2010/2011,1,1,1,C,2010-08-27,NA,0,2015/2016,130909,Academy sponsor led,Dixons City Academy,Not applicable


In [102]:
ever_AP <- enrol_final |>
    filter(InstitutionTypeDesc == 'AP/PRU') 

In [103]:
ever_AP_ids <- ever_AP |>
    select(person_id) |>
    distinct() |>
    mutate(ever_AP = TRUE)

In [104]:
ever_AP_ids

person_id,ever_AP
<chr>,<lgl>
B891969D99CBDDFD25843650004166B78065AA19286746C38AC2C2BDCC9DF517,TRUE
8B0EF0EF12A6FC021D4C25CEEEFE5B40E652B6391509975634A16AE4693F9EED,TRUE
2FD320DE2176D0875426B78D75636E1B998C566B5C6C5BE10686578EEC52ADD8,TRUE
09CAD5FFBCC56CEF7C8F4F6B461E0B27AF9B4432AF9F2616069CB27BB5F16611,TRUE
2C160E1145CBD3E2FC154D967298692B14D4623286F73E86F2970B408EBF018C,TRUE
1E430896DDAE48DE8494FF0C06E5A3C5D385456DE311B49F7C155BA3502D86DE,TRUE
C37FE6B9922AE18F48BC8ED4C583AD8373A292A3E83E71347364C34F43C96767,TRUE
13E07C3C09B0FA7310CA86A0ED66D2748561C27DA761E4552CF171D05D33DE3B,TRUE
5355B3AC36E9993749D3F2F86C889C52EF887A925F3BA43D6B5CBAF1CC6B074E,TRUE


In [105]:
write.csv(ever_AP_ids, "data/additional_ever_AP_ids.csv", row.names = FALSE)